In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
import sys

sys.path.append("..")

import meshio
from config import ROBOTIQ_HANDE_GRIPPER
from sampler import GraspSampler

from meshgraphnet.utils import msh_to_trimesh

msh = meshio.read("../meshes/primitives/msh/Cuboid1_cg1.msh")
mesh = msh_to_trimesh(msh)
# mesh = trimesh.load_mesh("../meshes/primitives/step/Cuboid1.STEP")
gripper = ROBOTIQ_HANDE_GRIPPER()
sampler = GraspSampler(mesh=mesh, gripper=gripper, mu=0.1)
grasps = sampler.sample(n_samples=1)



Sampled 1 antipodal grasps, checking for collisions...
1 valid grasps found after collision checking.


In [7]:
import sys

sys.path.append("..")

import meshio
import torch
from config import ROBOTIQ_HANDE_GRIPPER
from optimizer import GNNBasedGraspOptimizer

from meshgraphnet.nets import EncodeProcessDecode
from meshgraphnet.normalizer import Normalizer

gripper = ROBOTIQ_HANDE_GRIPPER()

msh = meshio.read("../meshes/primitives/msh/Cuboid1_cg1.msh")

checkpoint = torch.load(
    "../models/Mix_all_w.pth",
    map_location=torch.device("cpu"),
    weights_only=True,
)
model_state_dict = checkpoint["model_state_dict"]
params = checkpoint["params"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalizer = Normalizer(
    num_features=params["node_dim"],
    num_categorical=params["num_categorical"],
    device=device,
    stats=checkpoint["stats"],
)
model = EncodeProcessDecode(
    node_dim=params["node_dim"],
    edge_dim=params["edge_dim"],
    output_dim=params["output_dim"],
    latent_dim=params["latent_dim"],
    message_passing_steps=params["message_passing_steps"],
    use_layer_norm=params["use_layer_norm"],
).to(device)

optimizer = GNNBasedGraspOptimizer(gripper, model, normalizer, device=device)
grasp = optimizer.optimize(msh, mu=0.1, k=20)



Sampled 24 antipodal grasps, checking for collisions...
24 valid grasps found after collision checking.


In [8]:
sampler.visualize_grasp(grasp)